# HTB SMS preprocessing: watch each text stage

Module 292, Section 11. Use the **AI** kernel. This follows the exact course order—lowercase, regex cleaning, tokenization, stop-word removal, stemming, rejoin—on the UCI SMS Spam Collection. Named intermediate tables preserve the original text so you can see what changes. No model or text vectorizer is introduced yet.

## Reuse the Section 10 archive and exact-deduplicated starting table

Set `SMS_SPAM_ZIP` to a local UCI ZIP if present; otherwise fetch the same course link. Read only the expected archive member in memory. `quoting=csv.QUOTE_NONE` prevents literal quote marks inside SMS text from merging two message lines. An exact duplicate has the same label **and** message. This setup is carried forward from the preceding section, not a new modeling step.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import csv
import os
import re
import nltk
import pandas as pd
import requests

source_url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
local_zip = Path(os.environ.get('SMS_SPAM_ZIP', 'sms+spam+collection.zip'))
if local_zip.is_file():
    archive_bytes = local_zip.read_bytes()
else:
    response = requests.get(source_url, timeout=30)
    response.raise_for_status()
    archive_bytes = response.content
if len(archive_bytes) > 5_000_000:
    raise ValueError('Unexpectedly large SMS archive')
with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('SMSSpamCollection')
    if member.file_size > 2_000_000:
        raise ValueError('Unexpectedly large SMS text member')
    with archive.open(member) as source:
        raw_sms = pd.read_csv(source, sep='\t', header=None,
                              names=['label', 'message'], quoting=csv.QUOTE_NONE)
starting_sms = raw_sms.drop_duplicates().reset_index(drop=True)
assert starting_sms[['label', 'message']].notna().all().all()
assert starting_sms['label'].isin(['ham', 'spam']).all()
print('Rows after exact deduplication:', len(starting_sms))

## Check NLTK resources before processing

HTB downloads `punkt`, `punkt_tab`, and `stopwords`. `word_tokenize` needs the tokenizer data; `stopwords.words('english')` needs the English list. This cell checks local installation first, downloading only a missing resource. All three were already installed during validation in the **AI** environment. The download is a dependency step, not text cleaning itself.

In [ ]:
resources = {'tokenizers/punkt': 'punkt',
             'tokenizers/punkt_tab': 'punkt_tab',
             'corpora/stopwords': 'stopwords'}
for resource_path, package_name in resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        if not nltk.download(package_name, quiet=True, raise_on_error=True):
            raise RuntimeError(f'Could not obtain NLTK resource: {package_name}')
print('NLTK tokenizer and English stop-word resources are ready')

## 1. Lowercase

`FREE` and `free` become one spelling, lowering vocabulary size. `starting_sms` stays unchanged; `lower_sms` is a copy. This operation is deterministic but cannot be undone without the original. We show a small preview rather than the entire corpus.

In [ ]:
lower_sms = starting_sms.copy()
lower_sms['message'] = lower_sms['message'].str.lower()
print(lower_sms[['label', 'message']].head(3))
assert lower_sms['label'].equals(starting_sms['label'])
assert lower_sms['message'].map(lambda text: text == text.lower()).all()

## 2. Keep letters, whitespace, `$`, and `!`

HTB's regex `[^a-z\s$!]` reads **any character not among lowercase a through z, whitespace, dollar sign, or exclamation point**. `re.sub(..., '', text)` deletes every such character. For example, `FREE $20!!!` becomes `free $!!!`: the number vanishes but the symbols survive. Removing digits can also erase phone numbers, prices, and phishing clues, so this is a course demonstration rather than a universal cleanup rule.

In [ ]:
chars_sms = lower_sms.copy()
chars_sms['message'] = chars_sms['message'].map(
    lambda text: re.sub(r'[^a-z\s$!]', '', text))
print(chars_sms[['label', 'message']].head(3))
assert re.sub(r'[^a-z\s$!]', '', 'free $20!!!') == 'free $!!!'
assert chars_sms['label'].equals(lower_sms['label'])

## 3. Tokenize into lists

`word_tokenize` turns each string into a list of smaller units. A unit may be punctuation such as `!`, not just a word. HTB keeps the exclamation mark because it may convey spammy emphasis. From this point through stemming, `message` contains lists rather than strings, so code expecting raw text would have the wrong type.

In [ ]:
from nltk.tokenize import word_tokenize

tokens_sms = chars_sms.copy()
tokens_sms['message'] = tokens_sms['message'].map(word_tokenize)
print(tokens_sms[['label', 'message']].head(3))
assert tokens_sms['message'].map(lambda value: isinstance(value, list)).all()
assert word_tokenize('free prize!') == ['free', 'prize', '!']

## 4. Remove common stop words

A stop-word set contains frequent English items such as `the` and `and`. The list comprehension retains a token only if it is **not** in the set. This saves feature space but can lose negation or other context; check whether `not` appears in the installed set rather than assuming it is harmless to remove. Symbols retained above are not ordinary English stop words.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
no_stop_sms = tokens_sms.copy()
no_stop_sms['message'] = no_stop_sms['message'].map(
    lambda tokens: [token for token in tokens if token not in stop_words])
print(no_stop_sms[['label', 'message']].head(3))
print('Is not a stop word in this NLTK list?', 'not' in stop_words)
assert no_stop_sms['message'].map(
    lambda tokens: all(token not in stop_words for token in tokens)).all()

## 5. Apply Porter stemming

A stemmer mechanically folds related spellings toward a common stem. `running` → `run` is a simple example; other outputs may be fragments, not dictionary words. This can reduce feature count but also merge distinctions, so whether it helps is an empirical question for a later held-out evaluation.

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stemmed_sms = no_stop_sms.copy()
stemmed_sms['message'] = stemmed_sms['message'].map(
    lambda tokens: [stemmer.stem(token) for token in tokens])
print(stemmed_sms[['label', 'message']].head(3))
assert stemmer.stem('running') == 'run'
assert stemmed_sms['message'].map(lambda value: isinstance(value, list)).all()

## 6. Join tokens back into strings

`' '.join(tokens)` places one space between the remaining tokens, producing the string format a later vectorizer can read. This does **not** restore removed digits, punctuation, stop words, or capitalization. Keep `starting_sms` for provenance. No text vectorization or model fit is performed in this section.

In [ ]:
processed_sms = stemmed_sms.copy()
processed_sms['message'] = processed_sms['message'].map(' '.join)
print(processed_sms[['label', 'message']].head(3))
assert processed_sms['message'].map(lambda value: isinstance(value, str)).all()
assert processed_sms['label'].equals(starting_sms['label'])
assert len(processed_sms) == len(starting_sms)
print('Rows preserved through all text stages:', len(processed_sms))

The section has produced normalized strings and preserved labels. A real spam filter must apply the *same* text pipeline at training and prediction time. HTB's character and stop-word choices can remove valuable phishing evidence, so their effect on precision/recall must eventually be tested rather than assumed beneficial. The next course material, not this notebook, handles turning strings into numeric model features.